In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import h5py
import os
from pathlib import Path
from tqdm.auto import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.decomposition import PCA
from sklearn.model_selection import StratifiedKFold, LeaveOneOut
from sklearn.metrics import roc_auc_score, average_precision_score, roc_curve
from sklearn.cluster import KMeans
from scipy.stats import fisher_exact, mannwhitneyu
from statsmodels.stats.multitest import multipletests

import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams.update({'font.size': 12})
import warnings
warnings.filterwarnings('ignore')

FIG_DIR = '../figs'
os.makedirs(FIG_DIR, exist_ok=True)

# ── Program names (40 GEP programs) ─────────────────────────────────────
PROGRAM_NAMES = [
    'HALLMARK_ANGIOGENESIS', 'HALLMARK_APOPTOSIS', 'HALLMARK_COAGULATION',
    'HALLMARK_DNA_REPAIR', 'HALLMARK_E2F_TARGETS',
    'HALLMARK_EPITHELIAL_MESENCHYMAL_TRANSITION', 'HALLMARK_G2M_CHECKPOINT',
    'HALLMARK_GLYCOLYSIS', 'HALLMARK_HYPOXIA',
    'HALLMARK_IL6_JAK_STAT3_SIGNALING', 'HALLMARK_INFLAMMATORY_RESPONSE',
    'HALLMARK_INTERFERON_ALPHA_RESPONSE', 'HALLMARK_INTERFERON_GAMMA_RESPONSE',
    'HALLMARK_MTORC1_SIGNALING', 'HALLMARK_MYC_TARGETS_V1',
    'HALLMARK_OXIDATIVE_PHOSPHORYLATION', 'HALLMARK_TGF_BETA_SIGNALING',
    'HALLMARK_TNFA_SIGNALING_VIA_NFKB',
    'REACTOME_CLASS_I_MHC_MEDIATED_ANTIGEN_PROCESSING_PRESENTATION',
    'REACTOME_COLLAGEN_FORMATION', 'REACTOME_EXTRACELLULAR_MATRIX_ORGANIZATION',
    'REACTOME_INTEGRIN_CELL_SURFACE_INTERACTIONS',
    'REACTOME_MHC_CLASS_II_ANTIGEN_PRESENTATION', 'REACTOME_MISMATCH_REPAIR',
    'REACTOME_NEUTROPHIL_DEGRANULATION', 'REACTOME_TCR_SIGNALING',
    'REACTOME_TOLL_LIKE_RECEPTOR_CASCADES',
    'ANTIGEN_PRESENTATION_THOMPSON_2020_APM8', 'B_CELL_CORE_BUDCZIES_2021',
    'CIN70_CHROMOSOMAL_INSTABILITY', 'TGFb_STROMAL_EXCLUSION_MARIATHASAN_2018',
    'TLS_CABRITA_9', 'T_CELL_INFLAMED_GEP_18_AYERS_2017',
    'GOLDRATH_NAIVE_VS_MEMORY_CD8_TCELL_DN', 'GSE13306_TREG_VS_TCONV_UP',
    'GSE26495_PD1HIGH_VS_PD1LOW_CD8_TCELL_UP',
    'GSE5099_CLASSICAL_M1_VS_ALTERNATIVE_M2_MACROPHAGE_DN',
    'GSE5099_CLASSICAL_M1_VS_ALTERNATIVE_M2_MACROPHAGE_UP',
    'GSE9650_EFFECTOR_VS_EXHAUSTED_CD8_TCELL_UP',
    'GSE9946_IMMATURE_VS_MATURE_STIMULATORY_DC_DN',
]

# Clean scientific labels for plotting
PROGRAM_SHORT = ['Angiogenesis', 'Apoptosis', 'Coagulation', 'DNA Repair', 'E2F Targets',
                 'EMT', 'G2M Checkpoint', 'Glycolysis', 'Hypoxia', 'IL-6/JAK/STAT3', 
                 'Inflammatory Response', 'IFN-α Response', 'IFN-γ Response', 'mTORC1 Signaling', 
                 'MYC Targets', 'Oxidative Phosphorylation', 'TGF-β Signaling', 'TNF-α/NF-κB', 
                 'MHC-I Antigen Processing', 'Collagen Formation', 'ECM Organization', 'Integrin Interactions', 
                 'MHC-II Presentation', 'Mismatch Repair', 'Neutrophil Degranulation', 'TCR Signaling', 'TLR Cascades', 
                 'Antigen Presentation (APM)', 'B Cell Signature', 'Chromosomal Instability', 'TGF-β Stromal Exclusion', 
                 'Tertiary Lymphoid Structures', 'T Cell-Inflamed GEP', 'Naïve CD8⁺ T Cells', 'Treg Signature', 
                 'PD-1⁺ CD8⁺ T Cells', 'M2 Macrophages', 'M1 Macrophages', 'Exhausted CD8⁺ T Cells', 'Immature Dendritic Cells']

In [ ]:
# Paths
GEP_DIR = Path('/data/rbg/users/gcohn/trident/outputs_ovarian/20x_224px_0px_overlap/features_gep_40_flexible_normalized_robust_knn/')
EMB_DIR = Path('/data/rbg/users/gcohn/trident/outputs_ovarian/20x_224px_0px_overlap/features_hoptimus1')
CA125_XLS = Path('/data/rbg/users/azizayed/chemistry/scratch/ovarian/new_CA125-data_20230207.xlsx')

# Load metadata from both sheets (effective = responder, invalid = non-responder)
df_eff = pd.read_excel(CA125_XLS, sheet_name='Ovary.effective-162')
df_inv = pd.read_excel(CA125_XLS, sheet_name='Ovary.invalid-126')
df_eff['response_label'] = 1
df_inv['response_label'] = 0
df_inv = df_inv.rename(columns={'No.': 'No'})
meta_df = pd.concat([df_eff, df_inv], ignore_index=True)
meta_df['slide_id'] = meta_df['Image No.'].str.replace('.svs', '', regex=False, case=False)
meta_df['Patient ID'] = meta_df['Patient ID'].astype(str)

# Flag post-treatment slides
def is_post_treatment(slide_id):
    sid = str(slide_id).lower()
    return sid.startswith('2-p') or 'maintenance' in sid or sid.startswith('3-p')

meta_df['post_treatment'] = meta_df['slide_id'].apply(is_post_treatment)
n_post = meta_df['post_treatment'].sum()
print(f'Flagged {n_post} post-treatment slides (out of {len(meta_df)} total)')

# Exclude patients with mixed labels
label_per_pt = meta_df.groupby('Patient ID')['response_label'].nunique()
mixed_pts = label_per_pt[label_per_pt > 1].index.tolist()
print(f'Excluding {len(mixed_pts)} patients with mixed labels: {mixed_pts}')
meta_df = meta_df[~meta_df['Patient ID'].isin(mixed_pts)]

# Match with available features
gep_stems = {f.stem for f in GEP_DIR.glob('*.h5')}
emb_stems = {f.stem for f in EMB_DIR.glob('*.h5')}
meta_df = meta_df[meta_df['slide_id'].isin(gep_stems & emb_stems)]

# Patient-level labels
patient_label = meta_df.groupby('Patient ID')['response_label'].first()
patient_slides_pre = meta_df[~meta_df['post_treatment']].groupby('Patient ID')['slide_id'].apply(list)

print(f'\nPatients: {len(patient_label)}')
print(f'Responders: {patient_label.sum()} ({patient_label.mean():.1%})')
print(f'Pre-treatment slides: {(~meta_df["post_treatment"]).sum()}')

In [ ]:
# Load features — PRE-TREATMENT slides only
def read_h5(path, key='features'):
    with h5py.File(path, 'r') as f:
        return f[key][()]

def load_slide(sid):
    gep = read_h5(GEP_DIR / f'{sid}.h5', 'features').astype(np.float32)
    emb = read_h5(EMB_DIR / f'{sid}.h5', 'features').astype(np.float32)
    coords = read_h5(EMB_DIR / f'{sid}.h5', 'coords').astype(np.float32)
    return sid, gep, emb, coords

# Collect pre-treatment slide IDs
pid_to_slides = {}
all_slide_ids = []
for pid in patient_label.index:
    slides = patient_slides_pre.get(pid, [])
    if slides:
        pid_to_slides[pid] = slides
        all_slide_ids.extend(slides)

print(f'Loading {len(all_slide_ids)} slides for {len(pid_to_slides)} patients...')

slide_cache = {}
with ThreadPoolExecutor(max_workers=8) as executor:
    futures = {executor.submit(load_slide, sid): sid for sid in all_slide_ids}
    for future in tqdm(as_completed(futures), total=len(futures), desc='Loading'):
        sid, gep, emb, coords = future.result()
        slide_cache[sid] = (gep, emb, coords)

In [ ]:
# ── Aggregate to patient level: POOL all patches across slides, then summarize ──
rows = []
patient_patch_data = {}
skipped = []

for pid in patient_label.index:
    slides = pid_to_slides.get(pid, [])
    if not slides:
        skipped.append(pid)
        continue

    # Pool all patches across all slides for this patient
    all_gep = np.concatenate([slide_cache[sid][0] for sid in slides], axis=0)
    all_emb = np.concatenate([slide_cache[sid][1] for sid in slides], axis=0)

    # Store raw patch data for downstream use
    slide_data = [(sid, slide_cache[sid][0], slide_cache[sid][1], slide_cache[sid][2]) for sid in slides]
    patient_patch_data[pid] = slide_data

    # GEP: mean across all pooled patches (40d)
    gep_mean = all_gep.mean(axis=0)
    # EMB: mean across all pooled patches
    emb_mean = all_emb.mean(axis=0)

    row = {'patient_id': pid, 'response_label': patient_label[pid], 'n_slides': len(slides),
           'n_patches': all_gep.shape[0]}
    for i, v in enumerate(gep_mean): row[f'gep_{i}'] = v
    for i, v in enumerate(emb_mean): row[f'emb_{i}'] = v
    rows.append(row)

if skipped:
    print(f'Skipped {len(skipped)} patients with no pre-treatment slides')

patient_df = pd.DataFrame(rows)

gep_cols = [c for c in patient_df.columns if c.startswith('gep_')]
emb_cols = [c for c in patient_df.columns if c.startswith('emb_')]
X_gep_mean = patient_df[gep_cols].values   # (N, 40) — pooled mean
X_emb = patient_df[emb_cols].values         # (N, emb_dim)
y = patient_df['response_label'].values
patient_ids = patient_df['patient_id'].values

# Per-patient z-scored GEP (removes patient baseline, tests relative profile)
def zscore_per_patient(X):
    mu = X.mean(axis=1, keepdims=True)
    sd = X.std(axis=1, keepdims=True)
    sd[sd < 1e-8] = 1.0
    return (X - mu) / sd

X_gep_z = zscore_per_patient(X_gep_mean)  # (N, 40) — primary for MW

# PCA-reduced embeddings for combined features
X_emb_scaled = StandardScaler().fit_transform(X_emb)
X_emb_pca40 = PCA(n_components=40).fit_transform(X_emb_scaled)
X_gep = X_gep_mean  # alias for backward compat
X_combined = np.hstack([X_gep_mean, X_emb_pca40])

print(f'\nLoaded {len(patient_df)} patients')
print(f'X_gep_mean: {X_gep_mean.shape} (pooled patches, mean)')
print(f'X_gep_z: {X_gep_z.shape} (per-patient z-scored)')
print(f'X_emb: {X_emb.shape}')
print(f'X_combined: {X_combined.shape}')
print(f'Patches per patient: {patient_df["n_patches"].mean():.0f} mean, '
      f'{patient_df["n_patches"].median():.0f} median')
print(f'y: {y.sum()} responders / {len(y)} total ({y.mean():.1%})')

In [ ]:
# ── Per-program association with response (Mann-Whitney U, per-patient z-scored) ──
from scipy.stats import rankdata, norm
from statsmodels.stats.multitest import multipletests
from tqdm.auto import tqdm

def fast_mw_pvalues(X, y_vec):
    """Mann-Whitney U for all features using normal approximation."""
    mask = y_vec.astype(bool)
    n1, n2 = mask.sum(), (~mask).sum()
    ps, r_rbs = np.zeros(X.shape[1]), np.zeros(X.shape[1])
    for j in range(X.shape[1]):
        ranks = rankdata(X[:, j])
        U = ranks[mask].sum() - n1 * (n1 + 1) / 2
        mu_U = n1 * n2 / 2
        unique, counts = np.unique(ranks, return_counts=True)
        tie_corr = np.sum(counts**3 - counts) / (len(y_vec) * (len(y_vec) - 1))
        sigma_U = np.sqrt(n1 * n2 * ((len(y_vec) + 1) / 12 - tie_corr / 12))
        if sigma_U < 1e-10:
            ps[j], r_rbs[j] = 1.0, 0.0
            continue
        z = (U - mu_U) / sigma_U
        ps[j] = 2 * norm.sf(abs(z))
        r_rbs[j] = 2 * U / (n1 * n2) - 1
    return ps, r_rbs

def permutation_fdr(X, y_vec, n_perm=1000, seed=42):
    """Permutation-based FDR accounting for correlation between programs."""
    rng = np.random.RandomState(seed)
    obs_p, obs_r = fast_mw_pvalues(X, y_vec)
    null_p_matrix = np.zeros((n_perm, X.shape[1]))
    for i in tqdm(range(n_perm), desc='Permutation FDR', leave=False):
        null_p_matrix[i], _ = fast_mw_pvalues(X, rng.permutation(y_vec))
    sort_idx = np.argsort(obs_p)
    q_values = np.zeros(X.shape[1])
    for rank, j in enumerate(sort_idx):
        threshold = obs_p[j]
        n_obs_hits = (obs_p <= threshold).sum()
        mean_null_hits = np.mean(np.sum(null_p_matrix <= threshold, axis=1))
        q_values[j] = mean_null_hits / max(n_obs_hits, 1)
    for k in range(1, len(sort_idx)):
        q_values[sort_idx[k]] = max(q_values[sort_idx[k]], q_values[sort_idx[k - 1]])
    return obs_p, obs_r, np.minimum(q_values, 1.0)

# ── Run on per-patient z-scored pooled mean ──
n_programs = 40
resp_mask = (y == 1)
nonresp_mask = (y == 0)

obs_p, obs_r, q_perm = permutation_fdr(X_gep_z, y, n_perm=1000)
_, q_bh, _, _ = multipletests(obs_p, method='fdr_bh')

mw_df = pd.DataFrame({
    'program': [f'P{j}' for j in range(n_programs)],
    'idx': range(n_programs),
    'name': PROGRAM_NAMES,
    'short': PROGRAM_SHORT,
    'r_rb': obs_r,
    'p': obs_p,
    'p_fdr': q_bh,
    'q_perm': q_perm,
    'mean_resp': [X_gep_z[resp_mask, j].mean() for j in range(n_programs)],
    'mean_nonresp': [X_gep_z[nonresp_mask, j].mean() for j in range(n_programs)],
})

n_nom = (mw_df['p'] < 0.05).sum()
n_fdr = (mw_df['p_fdr'] < 0.05).sum()
n_perm = (mw_df['q_perm'] < 0.05).sum()

mw_df['sig'] = ''
mw_df.loc[mw_df['p'] < 0.05, 'sig'] = '*'
mw_df.loc[mw_df['q_perm'] < 0.05, 'sig'] = '**'
mw_df['neg_log10_p'] = -np.log10(mw_df['p'])

print(f'Mann-Whitney U (per-patient z-scored, pooled patches):')
print(f'  {resp_mask.sum()} resp vs {nonresp_mask.sum()} non-resp')
print(f'  Significant: {n_nom} nominal, {n_fdr} BH-FDR, {n_perm} perm-FDR\n')
print(mw_df.sort_values('p')[['short', 'r_rb', 'p', 'p_fdr', 'q_perm', 'sig']].to_string(index=False))
n_pos = (mw_df['r_rb'] > 0).sum()
n_neg = (mw_df['r_rb'] < 0).sum()
print(f'\nDirection: {n_pos}/40 programs higher in responders, {n_neg}/40 higher in non-responders')

In [ ]:
# ── Panel (a): Program Activation Heatmap (per-patient z-scored) ──────────
import seaborn as sns
from scipy.cluster.hierarchy import linkage, leaves_list
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch

# Use per-patient z-scored data (same as MW test) for visualization
X_heatmap = X_gep_z

# Sort patients: within-group hierarchical clustering
resp_idx = np.where(y == 1)[0]
nonresp_idx = np.where(y == 0)[0]

if len(resp_idx) > 2:
    resp_linkage = linkage(X_heatmap[resp_idx], method='ward')
    resp_sorted = resp_idx[leaves_list(resp_linkage)]
else:
    resp_sorted = resp_idx

if len(nonresp_idx) > 2:
    nonresp_linkage = linkage(X_heatmap[nonresp_idx], method='ward')
    nonresp_sorted = nonresp_idx[leaves_list(nonresp_linkage)]
else:
    nonresp_sorted = nonresp_idx

patient_order = np.concatenate([resp_sorted, nonresp_sorted])
X_sorted = X_heatmap[patient_order]
y_sorted = y[patient_order]

# Order programs alphabetically
prog_order = np.argsort(PROGRAM_SHORT)

row_labels = []
for j in prog_order:
    sig = mw_df.iloc[j]['sig']
    row_labels.append(f'{PROGRAM_SHORT[j]} {sig}'.rstrip())

n_resp = len(resp_sorted)
n_nonresp = len(nonresp_sorted)
N_total = n_resp + n_nonresp

fig = plt.figure(figsize=(16, 11))
gs = fig.add_gridspec(2, 2, height_ratios=[1, 30], width_ratios=[30, 1],
                       hspace=0.02, wspace=0.02)

ax_bar = fig.add_subplot(gs[0, 0])
bar_cmap = ListedColormap(['#e74c3c', '#3498db'])
ax_bar.imshow([y_sorted], aspect='auto', interpolation='nearest', cmap=bar_cmap, vmin=0, vmax=1)
ax_bar.set_xticks([]); ax_bar.set_yticks([]); ax_bar.tick_params(length=0)
ax_bar.axvline(n_resp - 0.5, color='black', linewidth=2)

legend_els = [Patch(facecolor='#3498db', label=f'Responder (n={n_resp})'),
              Patch(facecolor='#e74c3c', label=f'Non-responder (n={n_nonresp})')]
ax_bar.legend(handles=legend_els, loc='center left', fontsize=10,
              bbox_to_anchor=(1.02, 0.5), frameon=False)

ax_heat = fig.add_subplot(gs[1, 0])
im = ax_heat.imshow(X_sorted[:, prog_order].T, aspect='auto', cmap='RdBu_r',
                     vmin=-2.5, vmax=2.5, interpolation='nearest')
ax_heat.set_yticks(range(len(row_labels)))
ax_heat.set_yticklabels(row_labels, fontsize=8)
ax_heat.set_xticks([])
ax_heat.set_xlabel('Patients (hierarchically clustered within each group)', fontsize=11)
ax_heat.axvline(n_resp - 0.5, color='black', linewidth=2)

ax_cb = fig.add_subplot(gs[1, 1])
cb = plt.colorbar(im, cax=ax_cb)
cb.set_label('Per-patient z-scored activation', fontsize=10)

fig.suptitle(f'Predicted Gene Program Activation by Bevacizumab Response\n(Ovarian Cancer, N={N_total})',
             fontweight='bold', fontsize=14, y=0.98)

plt.savefig(f'{FIG_DIR}/ovarian_program_heatmap.pdf', bbox_inches='tight', dpi=150)
plt.savefig(f'{FIG_DIR}/ovarian_program_heatmap.png', bbox_inches='tight', dpi=600)
plt.show()

print(f'Heatmap: {n_resp} responders + {n_nonresp} non-responders = {N_total} total')
print(f'Significance: * p<0.05 (nominal), ** q<0.05 (perm-FDR)')
print(f'  {n_nom}/40 nominal, {n_perm}/40 perm-FDR')

## 4. Risk Stratification — Programs vs Embeddings

**Unsupervised**: Per-patient z-score → StandardScaler → PCA(10) → median split on PC1 → Fisher exact OR.
Fully unsupervised, balanced groups (n/2 vs n/2). Same preprocessing as supervised.

**Supervised**: Per-patient z-score → StandardScaler → PCA(10) → KNN(5), LOOCV → median-split → OR.  
Z-normalization captures relative program activation profiles. PCA(10) prevents overfitting given small sample size.

In [ ]:
# ── Risk Stratification: Unsupervised + Supervised ─────────────────────────
def compute_or_ci(groups, y, name, verbose=True):
    """OR with 95% CI (Woolf), Fisher exact p."""
    if y[groups == 1].mean() > y[groups == 0].mean():
        groups = 1 - groups
    a = ((groups == 0) & (y == 1)).sum()
    b = ((groups == 0) & (y == 0)).sum()
    c = ((groups == 1) & (y == 1)).sum()
    d = ((groups == 1) & (y == 0)).sum()
    _, p_val = fisher_exact([[a, b], [c, d]])
    aa, bb, cc, dd = (x + 0.5 if 0 in [a,b,c,d] else x for x in [a,b,c,d])
    or_val = (aa * dd) / (bb * cc)
    se = np.sqrt(1/aa + 1/bb + 1/cc + 1/dd)
    ci_lo = np.exp(np.log(or_val) - 1.96 * se)
    ci_hi = np.exp(np.log(or_val) + 1.96 * se)
    rate_0, rate_1 = y[groups == 0].mean(), y[groups == 1].mean()
    n_0, n_1 = (groups == 0).sum(), (groups == 1).sum()
    if verbose:
        print(f'{name}: OR={or_val:.2f} ({ci_lo:.2f}-{ci_hi:.2f}), p={p_val:.4f}  '
              f'[{rate_0:.0%} vs {rate_1:.0%}]  n={n_0}+{n_1}')
    return {'name': name, 'OR': or_val, 'CI_low': ci_lo, 'CI_high': ci_hi,
            'p': p_val, 'rate_0': rate_0, 'rate_1': rate_1, 'n_0': n_0, 'n_1': n_1}

def unsupervised_or(X, y, name, n_comp=10):
    """Unsupervised: per-patient z-score → Scale → PCA → median split on PC1 → OR."""
    mu = X.mean(axis=1, keepdims=True)
    sd = X.std(axis=1, keepdims=True)
    sd[sd < 1e-8] = 1.0
    X_z = (X - mu) / sd
    X_s = StandardScaler().fit_transform(X_z)
    X_pca = PCA(n_components=n_comp).fit_transform(X_s)
    pc1 = X_pca[:, 0]
    groups = (pc1 >= np.median(pc1)).astype(int)
    return compute_or_ci(groups, y, name)

def supervised_or(X, y, name, n_comp=10):
    """Supervised: per-patient z-score → Scale → PCA → KNN(5) LOOCV → median split → OR."""
    mu = X.mean(axis=1, keepdims=True)
    sd = X.std(axis=1, keepdims=True)
    sd[sd < 1e-8] = 1.0
    X_z = (X - mu) / sd

    loo = LeaveOneOut()
    oof = np.zeros(len(y))
    for train_idx, test_idx in loo.split(X_z):
        pipe = Pipeline([
            ('scale', StandardScaler()),
            ('pca', PCA(n_components=n_comp)),
            ('clf', KNeighborsClassifier(n_neighbors=5)),
        ])
        pipe.fit(X_z[train_idx], y[train_idx])
        oof[test_idx] = pipe.predict_proba(X_z[test_idx])[:, 1]

    auc = roc_auc_score(y, oof)
    groups = (oof >= np.median(oof)).astype(int)
    result = compute_or_ci(groups, y, name)
    result['AUC'] = auc
    result['oof'] = oof
    print(f'  (LOOCV AUC = {auc:.3f})')
    return result

print('=== Risk Stratification ===\n')
print('Unsupervised: per-patient z-score → PCA(10) → PC1 median split')
print('Supervised: per-patient z-score → PCA(10) → KNN(5) LOOCV\n')

or_results = []
or_results.append(unsupervised_or(X_gep_mean, y, 'GEP (unsupervised)', n_comp=10))
or_results.append(supervised_or(X_gep_mean, y, 'GEP (supervised)', n_comp=10))
or_results.append(unsupervised_or(X_emb, y, 'Embeddings (unsupervised)', n_comp=10))
or_results.append(supervised_or(X_emb, y, 'Embeddings (supervised)', n_comp=10))

or_df = pd.DataFrame([{k: v for k, v in r.items() if k != 'oof'} for r in or_results])
sup_oof = {r['name']: r['oof'] for r in or_results if 'oof' in r}

In [ ]:
# ── Forest plot: Supervised + Unsupervised OR ─────────────────────────────
# Sort by OR (highest at top)
plot_data = or_df.sort_values('OR', ascending=True).reset_index(drop=True)
y_pos = np.arange(len(plot_data))

# Color scheme consistent with external validation notebooks
color_map = {
    'GEP (unsupervised)':        '#B5CDE0',
    'GEP (supervised)':          '#2B6B8A',
    'Embeddings (unsupervised)': '#D1D1D1',
    'Embeddings (supervised)':   '#7A7A7A',
}

fig, ax = plt.subplots(figsize=(12, 5))

for i, (_, row) in enumerate(plot_data.iterrows()):
    color = color_map[row['name']]
    marker = 's' if 'supervised' in row['name'] else 'o'

    ax.errorbar(row['OR'], i,
                xerr=[[row['OR'] - row['CI_low']], [row['CI_high'] - row['OR']]],
                fmt=marker, color=color, markersize=10, capsize=5, linewidth=2.5,
                markeredgecolor='black', markeredgewidth=0.5)

ax.axvline(1, color='black', linestyle='--', linewidth=1, alpha=0.5)
ax.set_xscale('log')
ax.set_yticks(y_pos)
ax.set_yticklabels(plot_data['name'], fontsize=13)
ax.set_xlabel('Odds Ratio (95% CI)', fontsize=14)
ax.set_title('Risk Stratification: Predicted Programs vs Image Embeddings\n(Bevacizumab Response)',
             fontweight='bold', fontsize=15)

# Annotate OR + p to the right of each CI
for i, (_, row) in enumerate(plot_data.iterrows()):
    sig = ' *' if row['p'] < 0.05 else ''
    ax.annotate(f"OR={row['OR']:.2f} ({row['CI_low']:.2f}\u2013{row['CI_high']:.2f})  "
                f"p={row['p']:.3f}{sig}",
                xy=(row['CI_high'], i), xytext=(10, 0),
                textcoords='offset points', va='center', fontsize=11)

ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
ax.grid(axis='x', alpha=0.2)

# Legend below figure
from matplotlib.lines import Line2D
legend_els = [
    Line2D([0], [0], marker='o', color='w', markerfacecolor='#B5CDE0',
           markeredgecolor='black', markeredgewidth=0.5, markersize=9,
           label='GEP (unsupervised)'),
    Line2D([0], [0], marker='s', color='w', markerfacecolor='#2B6B8A',
           markeredgecolor='black', markeredgewidth=0.5, markersize=9,
           label='GEP (supervised)'),
    Line2D([0], [0], marker='o', color='w', markerfacecolor='#D1D1D1',
           markeredgecolor='black', markeredgewidth=0.5, markersize=9,
           label='Embeddings (unsupervised)'),
    Line2D([0], [0], marker='s', color='w', markerfacecolor='#7A7A7A',
           markeredgecolor='black', markeredgewidth=0.5, markersize=9,
           label='Embeddings (supervised)'),
]
fig.legend(handles=legend_els, loc='lower center', bbox_to_anchor=(0.5, -0.02),
           fontsize=10, ncol=4, frameon=False)

plt.subplots_adjust(bottom=0.22)
plt.savefig(f'{FIG_DIR}/ovarian_or_forest.pdf', bbox_inches='tight', dpi=150)
plt.savefig(f'{FIG_DIR}/ovarian_or_forest.png', bbox_inches='tight', dpi=600)
plt.show()